In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')
nltk.download('vader_lexicon')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")

print("Path to dataset files:", path)

In [ ]:
print(os.listdir(path))

In [ ]:
file_path = os.path.join(path, "Reviews.csv")

In [ ]:
df = pd.read_csv(file_path)

In [ ]:
df.head()

**EDA**

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df['Score']

In [ ]:
df['Score'].value_counts()

In [ ]:
ax = df['Score'].value_counts().sort_index()\
.plot(kind='bar', title='Count of Reviews by Stars', figsize=(10,5))
ax.set_xlabel('Reviews Stars')
ax.set_ylabel('Number of Reviews')
plt.show()

**NLTK Basics**

In [ ]:
example = df['Text'][10]
print(example)

In [ ]:
tokens = nltk.word_tokenize(example)
tokens[:10]

In [ ]:
tagged = nltk.pos_tag(tokens)
tagged[:10]

In [ ]:
entities = nltk.chunk.ne_chunk(tagged)
print(entities)

**STEP 1: VADER Sentiment Scoring**

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
from tqdm.notebook import tqdm

sia = SentimentIntensityAnalyzer()

In [ ]:
sia.polarity_scores(example)

In [ ]:
res = {}
for i, row in tqdm(df.iterrows(), total=len(df)):
  text = row['Text']
  myid = row['Id']
  res[myid] = sia.polarity_scores(text)

In [ ]:
list(res.items())[:10]

In [ ]:
vaders = pd.DataFrame(res).T
vaders = vaders.reset_index().rename(columns={"index":"Id"})
vaders = vaders.merge(df, how="left")

In [ ]:
vaders.head()

In [ ]:
sns.barplot(data=vaders, x='Score', y='compound')

**STEP 2: Roberta pretrained Model**

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from scipy.special import softmax

In [ ]:
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

In [ ]:
encoded_text = tokenizer(example, return_tensors='pt')
output = model(**encoded_text)
# output
scores = output[0][0].detach().numpy()
scores = softmax(scores)
scores_dict = {
    'roberta_neg': scores[0],
    'roberta_neu': scores[1],
    'roberta_pos': scores[2]
}
print(scores_dict)


In [ ]:
def polarity_score_roberta(example):
  encoded_text = tokenizer(example, return_tensors='pt')
  output = model(**encoded_text)
  scores = output[0][0].detach().numpy()
  scores = softmax(scores)
  scores_dict = {
      'roberta_neg': scores[0],
      'roberta_neu': scores[1],
      'roberta_pos': scores[2]
  }
  return scores_dict

In [ ]:
res = {}
for i, row in tqdm(df[:5000].iterrows(), total=5000):
  try:
    text = row['Text']
    myid = row['Id']
    vader_result = sia.polarity_scores(text)
    roberta_result = polarity_score_roberta(text)
    both = {**vader_result, **roberta_result}
    res[myid] = both
  except RuntimeError:
    print(f'Broke for id {myid}')

In [ ]:
result_df = pd.DataFrame(res).T
result_df = result_df.reset_index().rename(columns={"index":"Id"})
result_df = result_df.merge(df, how="left")

In [ ]:
result_df.head()

In [ ]:
result_df.columns

In [ ]:
sns.pairplot(data=result_df, vars=['neg', 'neu', 'pos', 'compound', 'roberta_neg', 'roberta_neu',
       'roberta_pos'],
       hue='Score', palette='tab10')